# CKM Analysis Notebook

This notebook reproduces the CKM pipeline, inspects recent commits, runs tests, computes diagnostics (including the Jarlskog invariant), and visualizes results.

In [ ]:
# 1. Environment & Imports

# Optional: install dev packages interactively
# %pip install --user pytest flake8 gitpython matplotlib scipy mypy

import sys, subprocess, json, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

try:
    from git import Repo
except Exception:
    Repo = None

print('Python', sys.version)
print('Repo root:', Path('.').resolve())

## 2. Inspect Repository & Latest Commits

# Show branch, status and recent commits using GitPython (if available) or subprocess

In [ ]:
import subprocess

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

print('Git status:')
print(run('git status --porcelain --branch').stdout)

print('\nRecent commits:')
print(run('git --no-pager log -n 10 --pretty=format:"%h %ad %s" --date=short').stdout)

# If GitPython is available, list commits with metadata
if Repo is not None:
    repo = Repo('.')
    print('\nGitPython branch:', repo.active_branch)
    for c in repo.iter_commits('HEAD', max_count=10):
        print(c.hexsha[:7], c.committed_datetime.isoformat(), c.author.name, c.message.strip())

## 3. List Files Changed in Recent Commits

# Collect files touched by the last N commits and print short diffs

In [ ]:
commits = subprocess.check_output(["git","log","-n","10","--pretty=format:%h"]).decode().split()
for c in commits:
    files = subprocess.check_output(["git","diff-tree","--no-commit-id","--name-only","-r",c]).decode().strip().splitlines()
    print(c, '->', files)
    # show short name-status for the commit
    print(subprocess.check_output(["git","show","--name-status","--pretty=format:'%h %s'",c]).decode()[:1000])


## 4. Run Test Suite & Capture Failures

# Run pytest programmatically and capture stdout/stderr; save outputs to `reports/pytest_last_output.txt`

In [ ]:
import sys
res = subprocess.run([sys.executable, '-m', 'pytest', '-q'], capture_output=True, text=True)
out = res.stdout + '\n' + res.stderr
Path('reports/pytest_last_output.txt').write_text(out)
print('Exit code:', res.returncode)
print(out[:2000])

# Try to extract failing test names
fails = []
for line in out.splitlines():
    if line.startswith('FAILED') or 'FAILURES' in line:
        fails.append(line)
print('Fail lines sample:', fails[:10])


## 5. Reproduce a Failing Case

# If tests reported failures, run them selectively with -k or run the target script directly

In [ ]:
# Example: run a single test by keyword
import sys
# replace <test_keyword> with the test name fragment
# subprocess.run([sys.executable, '-m', 'pytest', '-k', '<test_keyword>', '-q', '-vv'])

# Or run the test file directly
# subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_example.py::test_name', '-q', '-vv'])

print('Replace <test_keyword> above and re-run the cell to reproduce a failing case')

## 6. Debugging Session (local edits & pdb)

# Use pytest --capture=no for interactive debugging, or python -m pdb for step-through

# Example (uncomment to run):
# subprocess.run([sys.executable, '-m', 'pytest', '-k', '<test_keyword>', '--capture=no'])

print('Use the cells below to add print/debug statements or run pdb interactively.')

## 7. Implement Fix / Edit Source

# Minimal example: write a small patch to a file and re-run the failing test
from pathlib import Path

# Example: append a debug line (replace with a real patch)
# p = Path('some/module.py')
# txt = p.read_text()
# txt = txt.replace('buggy_call()', 'fixed_call()')
# p.write_text(txt)

print('Use the cell above to programmatically apply minimal fixes. Run tests after edits.')

In [ ]:
## 8. Add or Update Unit Tests

# Example: create a small test that verifies the global-fit JSON contains expected keys
from pathlib import Path
p = Path('data/ckm_global_fit.json')
if p.exists():
    d = json.loads(p.read_text())
    print('Keys in global fit JSON:', sorted(d.keys()))
    # Write a basic pytest that asserts presence of keys
    test_code = '''
import json
from pathlib import Path

def test_ckm_global_fit_keys():
    d = json.loads(Path('data/ckm_global_fit.json').read_text())
    assert 'left_scales' in d and 'right_scales' in d and 'unitary_real' in d
'''
    Path('tests/test_ckm_global_fit_basics.py').write_text(test_code)
    print('Wrote tests/test_ckm_global_fit_basics.py')
else:
    print('data/ckm_global_fit.json not found; run the optimizer first')

In [ ]:
## 9. Run Linters & Type Checks

# Run flake8 and mypy (if installed) and capture output
import shutil
if shutil.which('flake8'):
    print(subprocess.check_output(['flake8', '.']).decode()[:2000])
else:
    print('flake8 not installed; run `pip install flake8`')

if shutil.which('mypy'):
    print(subprocess.check_output(['mypy', '.']).decode()[:2000])
else:
    print('mypy not installed; skip type checks')

## 10. Commit, Push & Create Draft PR

# Stage, commit and push changes. Optionally create a draft PR using GitHub CLI `gh` if configured.

# Example commit (run when ready):
# subprocess.run(['git','add','-A'])
# subprocess.run(['git','commit','-m','"Fix: <short description>"'])
# subprocess.run(['git','push','origin','HEAD'])

print('When ready, stage and commit files. Use gh CLI to create a draft PR.')